# Semantic protest classifier evaluation

This notebook reads the JSON reports produced by `protest_classifier.evaluate`. It compares nested training releases on development data and summarizes the final, separately locked test evaluation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORTS = ROOT / 'reports'

def read_report(name):
    path = REPORTS / name
    if not path.exists():
        raise FileNotFoundError(f'Run evaluation first; missing {path}')
    return json.loads(path.read_text())

dev = read_report('new_classifier_dev.json')
test = read_report('new_classifier_test.json') if (REPORTS / 'new_classifier_test.json').exists() else None

## Overall metrics

In [ ]:
def overview(report):
    return pd.DataFrame([{
        'split': report['split'], 'run': row.get('run'), 'seed': row.get('seed'),
        'n': row['n'], 'strict_accuracy': row['strict_accuracy'],
        'macro_f1': row['macro_f1'], 'accepted_accuracy': row['accepted_accuracy'],
        'strict_95ci_low': row['strict_accuracy_95ci'][0],
        'strict_95ci_high': row['strict_accuracy_95ci'][1],
    } for row in report['models']])

tables = [overview(dev)] + ([overview(test)] if test else [])
pd.concat(tables, ignore_index=True).sort_values(['split', 'run', 'seed'])

## Development learning curve

In [ ]:
curve = overview(dev).copy()
curve['train_rows'] = curve['run'].str.replace('run-', '', regex=False).astype(int)
curve = curve.groupby('train_rows', as_index=False)[['strict_accuracy', 'macro_f1']].mean()
ax = curve.plot(x='train_rows', y=['strict_accuracy', 'macro_f1'], marker='o', figsize=(8, 4))
ax.set(xlabel='Training annotations', ylabel='Development score', ylim=(0, 1), title='Learning curve')
ax.grid(alpha=.25); plt.show()
curve

## Per-class performance and confusion matrix

In [ ]:
report = test if test else dev
best = max(report['models'], key=lambda row: row['macro_f1'])
per_class = pd.DataFrame(best['per_class']).T.sort_values('f1')
display(per_class)

matrix = np.asarray(best['confusion_matrix'])
labels = best['labels']
fig, ax = plt.subplots(figsize=(11, 10))
image = ax.imshow(matrix, cmap='Blues')
ax.set(xticks=range(len(labels)), yticks=range(len(labels)), xticklabels=labels, yticklabels=labels,
       xlabel='Predicted', ylabel='Gold', title=f"Confusion matrix: {report['split']} / {best.get('run')} / seed {best.get('seed')}")
plt.setp(ax.get_xticklabels(), rotation=90); fig.colorbar(image, ax=ax); fig.tight_layout(); plt.show()

## Concrete predictions

In [ ]:
print('Correct examples')
display(pd.DataFrame(best['examples']['correct']))
print('Errors to inspect')
display(pd.DataFrame(best['examples']['errors']))